# 01 - BB84 ideale


Simulazione del protocollo BB84 ideale usando le funzioni implementate in `src`.

Il caso studiato non include Eve né rumore di canale. Dopo il sifting, il QBER atteso sulla chiave sifted è 0.

Il sifting conserva solo i round in cui Alice e Bob hanno scelto la stessa base di misura.

## Setup

Import dei moduli del progetto e cartelle per tabelle e figure.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

current_path = Path.cwd()

if (current_path / "src" / "bb84.py").exists():
    project_root = current_path
else:
    project_root = current_path.parent

src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from bb84 import run_bb84_protocol
from qkd_core import sift_keys, compute_qber
from metrics import compute_sifted_key_length, compute_sifted_key_rate

tables_dir = project_root / "results" / "tables"
figures_dir = project_root / "results" / "figures"

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Setup completato.")

## Parametri

Impostiamo il numero di round e il seed della simulazione.

In [ ]:
N_ROUNDS = 1000
SEED = 42

print("Numero di round:", N_ROUNDS)
print("Seed:", SEED)

## Esecuzione del protocollo

Simulazione BB84 ideale, sifting e calcolo del QBER sulla chiave sifted.

In [ ]:
results = run_bb84_protocol(N_ROUNDS, seed=SEED)

alice_key, bob_key = sift_keys(results)
qber = compute_qber(alice_key, bob_key)

print("Lunghezza chiave Alice dopo sifting:", len(alice_key))
print("Lunghezza chiave Bob dopo sifting:", len(bob_key))
print("QBER:", qber)

## Tabella dei round

Convertiamo la lista dei risultati in una tabella pandas e salviamo tutti i round in formato CSV.

In [ ]:
df = pd.DataFrame(results)

rounds_path = tables_dir / "bb84_ideal_rounds.csv"
df.to_csv(rounds_path, index=False)

print(f"Tabella dei round salvata in: {rounds_path}")
df.head(10)

## Sintesi numerica

Metriche principali della simulazione in forma tabellare.

In [ ]:
sifted_key_length = compute_sifted_key_length(alice_key)
sifted_key_rate = compute_sifted_key_rate(alice_key, N_ROUNDS)

summary = {
    "protocol": "BB84 ideal",
    "n_rounds": N_ROUNDS,
    "sifted_key_length": sifted_key_length,
    "sifted_key_rate": sifted_key_rate,
    "qber": qber,
    "seed": SEED,
}

summary_df = pd.DataFrame([summary])

summary_path = tables_dir / "bb84_ideal_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"Sintesi salvata in: {summary_path}")
summary_df

## Grafico keep/discard

Round mantenuti e scartati dopo il confronto delle basi.

In [ ]:
kept_rounds = 0
discarded_rounds = 0

for i in range(len(results)):
    if results[i]["keep"] == True:
        kept_rounds += 1
    else:
        discarded_rounds += 1

labels = ["Mantenuti", "Scartati"]
values = [kept_rounds, discarded_rounds]

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.ylabel("Numero di round")
plt.title("BB84 ideale: round mantenuti e scartati")
plt.grid(axis="y")

figure_path = figures_dir / "bb84_ideal_keep_discard.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Round mantenuti:", kept_rounds)
print("Round scartati:", discarded_rounds)
print(f"Grafico salvato in: {figure_path}")

## Commento finale

Nel caso BB84 ideale il QBER sulla chiave sifted è nullo, perché non sono presenti né Eve né rumore di canale.

Circa metà dei round viene mantenuta dal sifting, dato che Alice e Bob scelgono casualmente tra due basi possibili, `Z` e `X`.

Questi risultati sono il riferimento per confrontare BB84 con Eve e BB84 in presenza di rumore.